#### Import python libraries

In [ ]:
!pip install tensorflow

In [19]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import random
import cv2
from datasets import load_dataset
import subprocess, tempfile, sys


In [20]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))

In [36]:
# Download the official annotation CSV — this is just a few MB
train_csv = pd.read_csv('/train.csv')

CHORE_CLASSES = [
    'washing dishes', 'vacuuming floor', 'mopping floor',
    'doing laundry',  'ironing clothes', 'folding clothes',
    'making bed',     'cleaning toilet', 'sweeping floor',
    'washing hands'
]

chores_df = train_csv[train_csv['label'].isin(CHORE_CLASSES)].reset_index(drop=True)
print(f'Found {len(chores_df)} chore clips across {chores_df["label"].nunique()} classes')
print(chores_df.head())


Found 3776 chore clips across 8 classes
             label   youtube_id  time_start  time_end  split
0       making bed  -0mnCHRQ-Zc          92       102  train
1       making bed  -JdIM1KZ1zo          32        42  train
2  folding clothes  -KtT7Q730Yg           8        18  train
3    washing hands  -LUN6528w3I          28        38  train
4  cleaning toilet  -P8Hq7Nc_lQ          93       103  train


##### Set Display Options for pandas

In [22]:
pd.set_option('display.max_colwidth', None)

#### Let us do Feature extraction

In [26]:
NUMBER_OF_FRAMES = 16
FRAME_WIDTH = 112
FRAME_HEIGHT = 112
BATCH_SIZE  = 8
EPOCHS = 20
LR = 1e-4
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32) # [R, G, B] - MobileNetV2 was pretrained on ImageNet using these RGB Channels
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32) # [R, G, B] - MobileNetV2 was pretrained on ImageNet using these RGB Channels

In [24]:
def extract_frames(video_path):
    captured_video = cv2.VideoCapture(video_path)
    total_number_of_frames = int(captured_video.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Total Number of frames: {total_number_of_frames}")
    indices = np.linspace(0, total_number_of_frames - 1, NUMBER_OF_FRAMES, dtype=int) # Uniform Temporal Sampling
    frames = []
    for index in indices:
        captured_video.set(cv2.CAP_PROP_POS_FRAMES, int(index))
        ok, frame = captured_video.read() # read_succeeded?, actual_pixel_data as numpy array of shape (H, W, 3)

        if not ok:
            frame = frames[-1].copy() if frames else np.zeros((FRAME_HEIGHT, FRAME_WIDTH, 3), dtype=np.uint8) # repeat the last successfully read frame if reading of current frame failed [maintains 16 frames]. If very first frame fails AND frames list is still empty, then i create blank frame as a fallback
        else:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (FRAME_WIDTH, FRAME_HEIGHT), interpolation=cv2.INTER_AREA) # resizes the frame from its original resolution down to 112×112.

        frames.append(frame)

    captured_video.release() # Close the video file and free the memory OpenCV allocated for the decoder.
    frames = np.stack(frames).astype(np.float32) / 255.0
    frames = (frames - IMAGENET_MEAN) / IMAGENET_STD
    return frames



In [23]:
def download_and_extract(row):
    """
    Stream one clip from YouTube using the row's video link and timestamps,
    extract frames, return numpy array.
    """
    url        = row['video link']
    start      = row['start_time']
    end        = start + row['duration']

    with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as f:
        tmp_path = f.name

    cmd = [
        sys.executable, '-m', 'yt_dlp',
        '--download-sections', f'*{start}-{end}',
        '--force-keyframes-at-cuts',
        '-f', 'mp4',
        '-o', tmp_path,
        '--quiet',
        url
    ]

    result = subprocess.run(cmd, capture_output=True)

    if result.returncode != 0 or not os.path.exists(tmp_path):
        print(f"Failed: {url}")
        return None

    try:
        frames = extract_frames(tmp_path)   # your existing function — unchanged
    finally:
        os.remove(tmp_path)                 # delete clip immediately after

    return frames

In [28]:
def make_dataset_from_urls(rows, labels, augment=False, batch_size=BATCH_SIZE):

    def load(row_idx, label):
        def _py(idx):
            idx    = int(idx.numpy())
            row    = rows[idx]
            frames = download_and_extract(row)

            if frames is None:                    # failed download fallback
                frames = np.zeros(
                    (NUMBER_OF_FRAMES, FRAME_HEIGHT, FRAME_WIDTH, 3), dtype=np.float32)

            return frames.astype(np.float32)

        frames = tf.py_function(_py, [row_idx], tf.float32)
        frames.set_shape([NUMBER_OF_FRAMES, FRAME_HEIGHT, FRAME_WIDTH, 3])

        if augment:
            frames = tf.cond(tf.random.uniform(()) > 0.5,
                             lambda: tf.image.flip_left_right(frames),
                             lambda: frames)
            frames = tf.image.random_brightness(frames, 0.1)
            frames = tf.image.random_contrast(frames, 0.85, 1.15)
            frames = tf.clip_by_value(frames, 0.0, 1.0)

        return frames, tf.cast(label, tf.int32)

    # Use integer indices since tf.data can't slice a HuggingFace dataset directly
    indices = list(range(len(rows)))
    ds = tf.data.Dataset.from_tensor_slices((indices, labels))

    if augment:
        ds = ds.shuffle(len(indices), reshuffle_each_iteration=True)

    return ds.map(load, num_parallel_calls=tf.data.AUTOTUNE) \
             .batch(batch_size) \
             .prefetch(tf.data.AUTOTUNE)

In [32]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Collect all rows as a list for indexing
# all_rows = list(chores_ds)

le = LabelEncoder()
encoded = le.fit_transform(all_labels)

X_tmp,   X_test,  y_tmp,   y_test  = train_test_split(
    all_rows, encoded, test_size=0.10, stratify=encoded, random_state=42)
X_train, X_val,   y_train, y_val   = train_test_split(
    X_tmp, y_tmp, test_size=0.15, stratify=y_tmp, random_state=42)

train_ds = make_dataset_from_urls(X_train, y_train, augment=True)
val_ds   = make_dataset_from_urls(X_val,   y_val,   augment=False)
test_ds  = make_dataset_from_urls(X_test,  y_test,  augment=False)

#### Model Training

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

NUM_CLASSES = len(le.classes_)

def build_model():
    backbone = keras.applications.MobileNetV2(
        input_shape=(FRAME_HEIGHT, FRAME_WIDTH, 3),
        include_top=False, weights='imagenet', pooling='avg'
    )
    backbone.trainable = False      # frozen in phase 1

    inp = keras.Input(shape=(NUMBER_OF_FRAMES, FRAME_HEIGHT, FRAME_WIDTH, 3))
    x = layers.TimeDistributed(backbone)(inp)           # CNN on every frame
    x = layers.TimeDistributed(layers.Dropout(0.3))(x)
    x = layers.LSTM(256, return_sequences=True)(x)      # temporal patterns
    x = layers.LSTM(128)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    return keras.Model(inp, out), backbone

model, backbone = build_model()
model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Phase 1 — frozen backbone
model.fit(train_ds, validation_data=val_ds, epochs=20,
          callbacks=[
              keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True),
              keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3),
              keras.callbacks.ModelCheckpoint('best.keras', save_best_only=True),
          ])

# Phase 2 — unfreeze top CNN layers and fine-tune
backbone.trainable = True
for layer in backbone.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5),   # 10× lower LR
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=10)

# Evaluate
loss, acc = model.evaluate(test_ds)
print(f'Test accuracy: {acc:.3f}')
model.save('action_classifier.keras')
np.save('label_classes.npy', le.classes_)